# 🌽 Treinamento do Modelo de Classificação de Pragas do Milho (MobileNetV2)

Este notebook executa o treinamento do modelo customizado MobileNetV2 utilizando aceleração por GPU no **Google Colab** e exporta o modelo otimizado para **TensorFlow Lite (`.tflite`)**.

---

### 1. Verificando o Acelerador de Hardware (GPU)
Certifique-se de que o ambiente de execução está configurado com GPU em: `Ambiente de execução` > `Alterar tipo de ambiente de execução` > `T4 GPU`.

In [ ]:
!nvidia-smi

### 2. Instalação das Dependências do Projeto

In [ ]:
# Instala as bibliotecas necessárias para treino e download
!pip install -q -r requirements.txt

### 3. Coleta de Imagens via GBIF (Opcional caso não tenha enviado dataset)
Você pode baixar fotos de campo de pragas diretamente pela internet de alta velocidade do Colab usando o script `01_baixar_gbif.py`:

In [ ]:
# Baixa as imagens de pragas diretamente para dataset/praga (ajuste o limite se desejar)
!python 01_baixar_gbif.py --max-por-classe 300 --saida dataset/praga --modo binario

### 4. Verificação da Estrutura de Classes do Dataset
Para a classificação binária funcionar, a pasta `dataset/` deve conter:
- `dataset/praga/`: Imagens das pragas do milho (ex.: lagartas e vaquinha).
- `dataset/saudavel/`: Imagens de folhas ou plantas sadias de controle.

In [ ]:
import os
from pathlib import Path

dataset_path = Path('dataset')
if dataset_path.exists():
    classes = [d.name for d in dataset_path.iterdir() if d.is_dir()]
    print(f'Pastas de classes encontradas: {classes}')
    for c in classes:
        total = len([f for f in (dataset_path / c).glob('*') if f.is_file() and not f.name.endswith('.txt')])
        print(f' - Classe "{c}": {total} imagens válidas')
    
    if 'saudavel' in classes and len(list((dataset_path / 'saudavel').glob('*.*'))) <= 1:
        print('\n⚠️ AVISO: A pasta dataset/saudavel está vazia ou contém apenas o README.')
        print('Envie fotos de folhas sadias para dataset/saudavel antes de iniciar o treinamento!')
else:
    print('Diretório dataset não encontrado. Verifique os arquivos enviados.')

### 5. Visualização de Amostras do Dataset & Data Augmentation

In [ ]:
# Gera as grades de visualização de lote e aumento de dados
!python src/corn_pest_classifier/visualizator.py

# Exibe os gráficos diretamente no Colab
from IPython.display import Image, display

if os.path.exists('preview_batch.png'):
    print('Lote original de treinamento:')
    display(Image('preview_batch.png'))

if os.path.exists('preview_augmentation.png'):
    print('Efeito das transformações de Data Augmentation:')
    display(Image('preview_augmentation.png'))

### 6. Treinamento da MobileNetV2 Customizada e Conversão para .tflite

In [ ]:
!python src/corn_pest_classifier/main.py

### 7. Download do Modelo Otimizado (.tflite)

In [ ]:
from google.colab import files
import os

modelo_tflite = 'milho_pragas_model.tflite'
if os.path.exists(modelo_tflite):
    print('Iniciando download do modelo .tflite...')
    files.download(modelo_tflite)
else:
    print('Arquivo .tflite não encontrado. Verifique se o treinamento concluiu com sucesso.')